# Генерация таблицы данных для M4 пайлайна

In [2]:
import sys
from dotenv import load_dotenv

sys.path.append('./../')
load_dotenv('./../.env')

True

In [3]:
import os
print(os.environ['OPENAI_API_KEY'])

YOUR_OPENAI_API_KEY


In [4]:
from typing import Any, List, TypedDict
import json
from langchain_openai import OpenAIEmbeddings
from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize
import numpy as np

from generators.m5_hard_dsl_dag_generator.dag_generator import DAGGenerationFlow, DAGGenerationState
from generators.m5_hard_dsl_dag_generator.md_generator import MDGenerationFlow, MDGenerationState

In [5]:
MODEL = 'gpt-5.4-mini'
MODEL_PROVIDER = 'openai'
TEMPERATURE = 0.5

DAG_GEN_PROMPT = './../prompts/md_generation/m5_hard_dsl_dag_gen/dag_generation.txt'
MD_GEN_PROMPT = './../prompts/md_generation/m5_hard_dsl_dag_gen/md_generation.txt'
MAX_RETRIES = 16
RUNS = 8

SAVING_DIR = './../data/dag_generations/'

def read_prompts(pr_path:str) -> str:
    with open(pr_path, 'rt') as fio:
        res = fio.read()
    return res 

In [6]:
import time
from dataclasses import dataclass
from typing import Optional
from langchain_core.callbacks import BaseCallbackHandler


class TokenCounter(BaseCallbackHandler):
    def __init__(self):
        self.prompt_tokens     = 0
        self.completion_tokens = 0
        self.total_tokens      = 0
        self.n_calls           = 0

    def on_llm_end(self, response, **kwargs):
        usage = None

        # LangChain wraps different providers differently
        if hasattr(response, "llm_output") and response.llm_output:
            usage = response.llm_output.get("token_usage") or \
                    response.llm_output.get("usage")

        # OpenAI via langchain_openai exposes it here
        if usage is None and response.generations:
            for gen_list in response.generations:
                for gen in gen_list:
                    info = getattr(gen, "generation_info", None) or {}
                    usage = info.get("usage") or info.get("token_usage")
                    if usage:
                        break

        if usage:
            self.prompt_tokens     += usage.get("prompt_tokens",     0)
            self.completion_tokens += usage.get("completion_tokens", 0)
            self.total_tokens      += usage.get("total_tokens",      0)
        self.n_calls += 1

In [7]:
class QueryDescr(TypedDict):
    id: str
    topic: str
    field_of_science: str
    llm_query: str

def read_tasks(tasks_file: str) -> List[str]:
    with open(tasks_file, 'rt') as fio:
        d = json.load(fio)
        queries_list = []
        for el in d:
            queries_list.append(QueryDescr(el))
    return queries_list

def generate_dag(md_gen_wf: DAGGenerationFlow, query: str):
    counter = TokenCounter()

    md_gen_state = DAGGenerationState()
    wf = md_gen_wf.workflow
    
    t0 = time.perf_counter()
    md_gen_state['users_task'] = query['llm_query']
    res_state = wf.invoke(
        md_gen_state,
        config={"callbacks": [counter],
            "recursion_limit":300},
    )
    elapsed = time.perf_counter() - t0

    tokens_dict = {
        'prompt': counter.prompt_tokens,
        'completion': counter.completion_tokens,
        'total': counter.total_tokens
    }

    return res_state, elapsed, tokens_dict

def generate_meta(md_gen_wf: MDGenerationFlow, query: str, md_dag):
    counter = TokenCounter()

    md_gen_state = MDGenerationState()
    wf = md_gen_wf.workflow
    
    t0 = time.perf_counter()
    md_gen_state['users_task'] = query['llm_query']
    md_gen_state['md_name'] = md_dag['raw_dag'].md_name
    md_gen_state['md_description'] = md_dag['raw_dag'].md_descr
    md_gen_state['dag'] = md_dag['dag']
    
    res_state = wf.invoke(
        md_gen_state,
        config={
            "callbacks": [counter],
            "recursion_limit":300
            },
    )
    elapsed = time.perf_counter() - t0

    tokens_dict = {
        'prompt': counter.prompt_tokens,
        'completion': counter.completion_tokens,
        'total': counter.total_tokens
    }

    return res_state, elapsed, tokens_dict


## Testing errors during per synthetic query

In [7]:
import json
from dag_viz import visualize_dag

os.makedirs(SAVING_DIR, exist_ok=True)

q_bank = read_tasks("./../queries/bio_queries.yaml")
all_stats = []
for q_idx, q in enumerate(q_bank):
    for r_idx in range(RUNS):
        save_dir = os.path.join(SAVING_DIR, f'q_{q_idx}_r_{r_idx}')
        os.makedirs(save_dir, exist_ok=True)

        dag_maker = DAGGenerationFlow(
            read_prompts(DAG_GEN_PROMPT), 
            MODEL, 
            MODEL_PROVIDER, 
            TEMPERATURE,
            MAX_RETRIES
        )
        md, el_t, tokens = generate_dag(dag_maker, q)

        print(f'DAG q={q_idx} r={r_idx} complete. Succes: {not md["generation_failed"]}. Tokens: {tokens["completion"]}, iters: {md["rework_attempts"] + 1}.')
        
        stats = {
            'q': q_idx,
            'r': r_idx,
            'tokens_prompt': tokens['prompt'],
            'tokens_compl': tokens['completion'],
            'tokes_total': tokens['total'],
            'time': el_t,
            'success': not md["generation_failed"],
            'all_errors': md['all_dag_errors'],
            'reworks': md['rework_attempts']
        }
        with open(os.path.join(save_dir, 'stats.json'), 'wt') as fo:
            json.dump(stats, fo)
        all_stats.append(stats)

        if not md["generation_failed"]:
            visualize_dag(md['dag'], os.path.join(save_dir, 'schema_vis.html'), open_browser=False)
            with open(os.path.join(save_dir, 'dag.json'), 'wt') as fo:
                json.dump(md['raw_dag'].model_dump_json(), fo)


DAG q=0 r=0 complete. Succes: True. Tokens: 5858, iters: 3.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_0_r_0/schema_vis.html
DAG q=0 r=1 complete. Succes: True. Tokens: 5403, iters: 3.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_0_r_1/schema_vis.html
DAG q=0 r=2 complete. Succes: True. Tokens: 3408, iters: 2.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_0_r_2/schema_vis.html
DAG q=0 r=3 complete. Succes: True. Tokens: 5380, iters: 3.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_0_r_3/schema_vis.html
DAG q=0 r=4 complete. Succes: True. Tokens: 10011, iters: 5.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_0_r_4/schema_vis.html
DAG q=0 r=5 complete. Succes: True. Tokens: 10831, iters: 5.
DAG saved → /

TypeError: 'bool' object is not iterable

In [ ]:
log_str = """
DAG q=0 r=0 complete. Succes: True. Tokens: 5858, iters: 3.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_0_r_0/schema_vis.html
DAG q=0 r=1 complete. Succes: True. Tokens: 5403, iters: 3.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_0_r_1/schema_vis.html
DAG q=0 r=2 complete. Succes: True. Tokens: 3408, iters: 2.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_0_r_2/schema_vis.html
DAG q=0 r=3 complete. Succes: True. Tokens: 5380, iters: 3.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_0_r_3/schema_vis.html
DAG q=0 r=4 complete. Succes: True. Tokens: 10011, iters: 5.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_0_r_4/schema_vis.html
DAG q=0 r=5 complete. Succes: True. Tokens: 10831, iters: 5.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_0_r_5/schema_vis.html
DAG q=0 r=6 complete. Succes: True. Tokens: 15716, iters: 8.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_0_r_6/schema_vis.html
DAG q=0 r=7 complete. Succes: True. Tokens: 5125, iters: 3.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_0_r_7/schema_vis.html
DAG q=1 r=0 complete. Succes: True. Tokens: 13829, iters: 4.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_1_r_0/schema_vis.html
DAG q=1 r=1 complete. Succes: True. Tokens: 11637, iters: 4.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_1_r_1/schema_vis.html
DAG q=1 r=2 complete. Succes: True. Tokens: 5644, iters: 2.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_1_r_2/schema_vis.html
DAG q=1 r=3 complete. Succes: True. Tokens: 13612, iters: 4.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_1_r_3/schema_vis.html
DAG q=1 r=4 complete. Succes: True. Tokens: 5658, iters: 2.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_1_r_4/schema_vis.html
DAG q=1 r=5 complete. Succes: True. Tokens: 8355, iters: 3.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_1_r_5/schema_vis.html
DAG q=1 r=6 complete. Succes: True. Tokens: 18148, iters: 6.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_1_r_6/schema_vis.html
DAG q=1 r=7 complete. Succes: True. Tokens: 15004, iters: 5.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_1_r_7/schema_vis.html
DAG q=2 r=0 complete. Succes: True. Tokens: 3820, iters: 2.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_2_r_0/schema_vis.html
DAG q=2 r=1 complete. Succes: True. Tokens: 3970, iters: 2.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_2_r_1/schema_vis.html
DAG q=2 r=2 complete. Succes: True. Tokens: 3910, iters: 2.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_2_r_2/schema_vis.html
DAG q=2 r=3 complete. Succes: True. Tokens: 5686, iters: 3.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_2_r_3/schema_vis.html
DAG q=2 r=4 complete. Succes: True. Tokens: 7347, iters: 4.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_2_r_4/schema_vis.html
DAG q=2 r=5 complete. Succes: True. Tokens: 8166, iters: 4.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_2_r_5/schema_vis.html
DAG q=2 r=6 complete. Succes: True. Tokens: 9175, iters: 5.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_2_r_6/schema_vis.html
DAG q=2 r=7 complete. Succes: True. Tokens: 4340, iters: 2.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_2_r_7/schema_vis.html
DAG q=3 r=0 complete. Succes: True. Tokens: 5731, iters: 2.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_3_r_0/schema_vis.html
DAG q=3 r=1 complete. Succes: True. Tokens: 4631, iters: 2.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_3_r_1/schema_vis.html
DAG q=3 r=2 complete. Succes: True. Tokens: 9673, iters: 4.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_3_r_2/schema_vis.html
DAG q=3 r=3 complete. Succes: True. Tokens: 14530, iters: 4.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_3_r_3/schema_vis.html
DAG q=3 r=4 complete. Succes: True. Tokens: 5458, iters: 2.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_3_r_4/schema_vis.html
DAG q=3 r=5 complete. Succes: True. Tokens: 7489, iters: 3.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_3_r_5/schema_vis.html
DAG q=3 r=6 complete. Succes: True. Tokens: 8780, iters: 3.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_3_r_6/schema_vis.html
DAG q=3 r=7 complete. Succes: False. Tokens: 50766, iters: 17.
DAG q=4 r=0 complete. Succes: True. Tokens: 3673, iters: 4.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_4_r_0/schema_vis.html
DAG q=4 r=1 complete. Succes: True. Tokens: 5406, iters: 4.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_4_r_1/schema_vis.html
DAG q=4 r=2 complete. Succes: True. Tokens: 3930, iters: 5.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_4_r_2/schema_vis.html
DAG q=4 r=3 complete. Succes: True. Tokens: 3180, iters: 6.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_4_r_3/schema_vis.html
DAG q=4 r=4 complete. Succes: True. Tokens: 2059, iters: 7.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_4_r_4/schema_vis.html
DAG q=4 r=5 complete. Succes: True. Tokens: 3401, iters: 9.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_4_r_5/schema_vis.html
DAG q=4 r=6 complete. Succes: True. Tokens: 3165, iters: 6.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_4_r_6/schema_vis.html
DAG q=4 r=7 complete. Succes: True. Tokens: 9583, iters: 12.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_4_r_7/schema_vis.html
DAG q=5 r=0 complete. Succes: True. Tokens: 9490, iters: 3.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_5_r_0/schema_vis.html
DAG q=5 r=1 complete. Succes: True. Tokens: 13281, iters: 3.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_5_r_1/schema_vis.html
DAG q=5 r=2 complete. Succes: True. Tokens: 6884, iters: 2.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_5_r_2/schema_vis.html
DAG q=5 r=3 complete. Succes: True. Tokens: 15423, iters: 5.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_5_r_3/schema_vis.html
DAG q=5 r=4 complete. Succes: True. Tokens: 6010, iters: 2.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_5_r_4/schema_vis.html
DAG q=5 r=5 complete. Succes: True. Tokens: 43808, iters: 10.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_5_r_5/schema_vis.html
DAG q=5 r=6 complete. Succes: True. Tokens: 5569, iters: 2.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_5_r_6/schema_vis.html
DAG q=5 r=7 complete. Succes: True. Tokens: 4822, iters: 2.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_5_r_7/schema_vis.html
DAG q=6 r=0 complete. Succes: True. Tokens: 5691, iters: 2.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_6_r_0/schema_vis.html
DAG q=6 r=1 complete. Succes: True. Tokens: 6459, iters: 2.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_6_r_1/schema_vis.html
DAG q=6 r=2 complete. Succes: True. Tokens: 10309, iters: 3.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_6_r_2/schema_vis.html
DAG q=6 r=3 complete. Succes: True. Tokens: 5550, iters: 2.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_6_r_3/schema_vis.html
DAG q=6 r=4 complete. Succes: True. Tokens: 7270, iters: 2.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_6_r_4/schema_vis.html
DAG q=6 r=5 complete. Succes: True. Tokens: 6140, iters: 2.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_6_r_5/schema_vis.html
DAG q=6 r=6 complete. Succes: True. Tokens: 14712, iters: 2.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_6_r_6/schema_vis.html
DAG q=6 r=7 complete. Succes: True. Tokens: 7384, iters: 2.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_6_r_7/schema_vis.html
DAG q=7 r=0 complete. Succes: True. Tokens: 7228, iters: 3.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_7_r_0/schema_vis.html
DAG q=7 r=1 complete. Succes: True. Tokens: 16508, iters: 7.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_7_r_1/schema_vis.html
DAG q=7 r=2 complete. Succes: True. Tokens: 7502, iters: 3.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_7_r_2/schema_vis.html
DAG q=7 r=3 complete. Succes: True. Tokens: 15217, iters: 6.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_7_r_3/schema_vis.html
DAG q=7 r=4 complete. Succes: True. Tokens: 6844, iters: 3.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_7_r_4/schema_vis.html
DAG q=7 r=5 complete. Succes: True. Tokens: 5727, iters: 3.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_7_r_5/schema_vis.html
DAG q=7 r=6 complete. Succes: True. Tokens: 4394, iters: 2.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_7_r_6/schema_vis.html
DAG q=7 r=7 complete. Succes: True. Tokens: 12335, iters: 5.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_7_r_7/schema_vis.html
DAG q=8 r=0 complete. Succes: True. Tokens: 13253, iters: 4.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_8_r_0/schema_vis.html
DAG q=8 r=1 complete. Succes: True. Tokens: 6681, iters: 2.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_8_r_1/schema_vis.html
DAG q=8 r=2 complete. Succes: True. Tokens: 6678, iters: 2.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_8_r_2/schema_vis.html
DAG q=8 r=3 complete. Succes: True. Tokens: 6503, iters: 2.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_8_r_3/schema_vis.html
DAG q=8 r=4 complete. Succes: True. Tokens: 9765, iters: 3.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_8_r_4/schema_vis.html
DAG q=8 r=5 complete. Succes: True. Tokens: 6232, iters: 2.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_8_r_5/schema_vis.html
DAG q=8 r=6 complete. Succes: True. Tokens: 6887, iters: 2.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_8_r_6/schema_vis.html
DAG q=8 r=7 complete. Succes: True. Tokens: 16907, iters: 5.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_8_r_7/schema_vis.html
DAG q=9 r=0 complete. Succes: True. Tokens: 5423, iters: 2.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_9_r_0/schema_vis.html
DAG q=9 r=1 complete. Succes: True. Tokens: 7854, iters: 3.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_9_r_1/schema_vis.html
DAG q=9 r=2 complete. Succes: True. Tokens: 5356, iters: 2.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_9_r_2/schema_vis.html
DAG q=9 r=3 complete. Succes: True. Tokens: 6620, iters: 3.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_9_r_3/schema_vis.html
DAG q=9 r=4 complete. Succes: True. Tokens: 8347, iters: 4.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_9_r_4/schema_vis.html
DAG q=9 r=5 complete. Succes: True. Tokens: 3912, iters: 2.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_9_r_5/schema_vis.html
DAG q=9 r=6 complete. Succes: True. Tokens: 6939, iters: 3.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_9_r_6/schema_vis.html
DAG q=9 r=7 complete. Succes: True. Tokens: 10189, iters: 4.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_9_r_7/schema_vis.html
DAG q=10 r=0 complete. Succes: True. Tokens: 13005, iters: 5.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_10_r_0/schema_vis.html
DAG q=10 r=1 complete. Succes: True. Tokens: 3853, iters: 2.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_10_r_1/schema_vis.html
DAG q=10 r=2 complete. Succes: True. Tokens: 4667, iters: 2.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_10_r_2/schema_vis.html
DAG q=10 r=3 complete. Succes: True. Tokens: 7682, iters: 4.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_10_r_3/schema_vis.html
DAG q=10 r=4 complete. Succes: True. Tokens: 6756, iters: 3.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_10_r_4/schema_vis.html
DAG q=10 r=5 complete. Succes: True. Tokens: 6035, iters: 3.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_10_r_5/schema_vis.html
DAG q=10 r=6 complete. Succes: True. Tokens: 4822, iters: 3.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_10_r_6/schema_vis.html
DAG q=10 r=7 complete. Succes: True. Tokens: 7871, iters: 3.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_10_r_7/schema_vis.html
DAG q=11 r=0 complete. Succes: True. Tokens: 7828, iters: 3.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_11_r_0/schema_vis.html
DAG q=11 r=1 complete. Succes: True. Tokens: 10973, iters: 5.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_11_r_1/schema_vis.html
DAG q=11 r=2 complete. Succes: True. Tokens: 6844, iters: 3.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_11_r_2/schema_vis.html
DAG q=11 r=3 complete. Succes: True. Tokens: 6348, iters: 3.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_11_r_3/schema_vis.html
DAG q=11 r=4 complete. Succes: True. Tokens: 8069, iters: 4.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_11_r_4/schema_vis.html
DAG q=11 r=5 complete. Succes: True. Tokens: 10591, iters: 4.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_11_r_5/schema_vis.html
DAG q=11 r=6 complete. Succes: True. Tokens: 5494, iters: 2.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_11_r_6/schema_vis.html
DAG q=11 r=7 complete. Succes: True. Tokens: 13058, iters: 5.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_11_r_7/schema_vis.html
DAG q=12 r=0 complete. Succes: True. Tokens: 4770, iters: 2.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_12_r_0/schema_vis.html
DAG q=12 r=1 complete. Succes: True. Tokens: 6135, iters: 3.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_12_r_1/schema_vis.html
DAG q=12 r=2 complete. Succes: True. Tokens: 5738, iters: 2.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_12_r_2/schema_vis.html
DAG q=12 r=3 complete. Succes: True. Tokens: 6899, iters: 3.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_12_r_3/schema_vis.html
DAG q=12 r=4 complete. Succes: True. Tokens: 13186, iters: 5.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_12_r_4/schema_vis.html
DAG q=12 r=5 complete. Succes: True. Tokens: 4255, iters: 2.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_12_r_5/schema_vis.html
DAG q=12 r=6 complete. Succes: True. Tokens: 12971, iters: 5.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_12_r_6/schema_vis.html
DAG q=12 r=7 complete. Succes: True. Tokens: 20227, iters: 7.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_12_r_7/schema_vis.html
DAG q=13 r=0 complete. Succes: True. Tokens: 15479, iters: 5.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_13_r_0/schema_vis.html
DAG q=13 r=1 complete. Succes: True. Tokens: 11335, iters: 4.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_13_r_1/schema_vis.html
DAG q=13 r=2 complete. Succes: True. Tokens: 6267, iters: 2.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_13_r_2/schema_vis.html
DAG q=13 r=3 complete. Succes: True. Tokens: 9265, iters: 3.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_13_r_3/schema_vis.html
DAG q=13 r=4 complete. Succes: True. Tokens: 6134, iters: 2.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_13_r_4/schema_vis.html
DAG q=13 r=5 complete. Succes: True. Tokens: 5740, iters: 2.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_13_r_5/schema_vis.html
DAG q=13 r=6 complete. Succes: True. Tokens: 12783, iters: 4.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_13_r_6/schema_vis.html
DAG q=13 r=7 complete. Succes: True. Tokens: 8888, iters: 4.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_13_r_7/schema_vis.html
DAG q=14 r=0 complete. Succes: True. Tokens: 7864, iters: 3.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_14_r_0/schema_vis.html
DAG q=14 r=1 complete. Succes: True. Tokens: 5520, iters: 2.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_14_r_1/schema_vis.html
DAG q=14 r=2 complete. Succes: True. Tokens: 4456, iters: 2.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_14_r_2/schema_vis.html
DAG q=14 r=3 complete. Succes: True. Tokens: 6383, iters: 2.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_14_r_3/schema_vis.html
DAG q=14 r=4 complete. Succes: True. Tokens: 6511, iters: 3.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_14_r_4/schema_vis.html
DAG q=14 r=5 complete. Succes: True. Tokens: 12435, iters: 4.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_14_r_5/schema_vis.html
DAG q=14 r=6 complete. Succes: True. Tokens: 8525, iters: 3.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_14_r_6/schema_vis.html
DAG q=14 r=7 complete. Succes: True. Tokens: 7165, iters: 3.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_14_r_7/schema_vis.html
DAG q=15 r=0 complete. Succes: True. Tokens: 4914, iters: 4.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_15_r_0/schema_vis.html
DAG q=15 r=1 complete. Succes: True. Tokens: 4773, iters: 6.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_15_r_1/schema_vis.html
DAG q=15 r=2 complete. Succes: True. Tokens: 4290, iters: 5.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_15_r_2/schema_vis.html
DAG q=15 r=3 complete. Succes: True. Tokens: 7727, iters: 6.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_15_r_3/schema_vis.html
DAG q=15 r=4 complete. Succes: True. Tokens: 9113, iters: 5.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_15_r_4/schema_vis.html
DAG q=15 r=5 complete. Succes: True. Tokens: 7770, iters: 7.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_15_r_5/schema_vis.html
DAG q=15 r=6 complete. Succes: True. Tokens: 2628, iters: 7.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_15_r_6/schema_vis.html
DAG q=15 r=7 complete. Succes: True. Tokens: 9391, iters: 4.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/dag_generations/q_15_r_7/schema_vis.html
"""

reg_exp = "iters: \d*"

import re
elems = re.findall(reg_exp, log_str)
elems = [int(e.split(' ')[-1]) for e in elems]

print(np.mean(elems), np.std(elems))
3.703125 2.1373149590958747

3.703125 2.1373149590958747


In [ ]:
import pandas as pd
df = pd.DataFrame(all_stats)
df.to_csv(os.path.join(SAVING_DIR, 'all_stats.csv'))

In [ ]:
df.head()